In [3]:
import heapq

def image_segmentation_graph_cut(image, source_weights, sink_weights, smoothness=1.0):
  rows = len(image)
  cols = len(image[0])

  SOURCE = rows*cols
  SINK = rows*cols + 1
  N = rows*cols + 2

  capacity = [[0.0] * N for _ in range(N)]

  def add_edge(u,v,cap):
    capacity[u][v] += cap

  for r in range(rows):
    for c in range(cols):
      node = r*cols + c
      # Changed 'add_edges' to 'add_edge' here
      add_edge(SOURCE, node, source_weights[r][c])
      add_edge(node, SINK, sink_weights[r][c])

  def boundary_penalty(i1,i2):
    diff = abs(i1 - i2)
    return smoothness * (1.0 / (1.0 + diff))

  for r in range(rows):
    for c in range(cols):
      node = r * cols + c
      for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols:
          neighbor = nr * cols + nc # Corrected calculation of 'neighbor' node index
          w = boundary_penalty(image[r][c], image[nr][nc])
          add_edge(node, neighbor, w)
          add_edge(neighbor, node, w)

  def bfs(source, sink, parent):
    visited = [False] * N
    visited[source] = True
    queue = [source]
    while queue:
      u = queue.pop(0) # Changed U to u to match usage below
      for v in range(N):
        if not visited[v] and capacity[u][v] > 1e-9:
          visited[v] = True
          parent[v] = u
          if v == sink:
            return True
          queue.append(v)
    return False

  parent = [-1] * N
  while bfs(SOURCE, SINK, parent):
    path_flow = float('inf')
    v = SINK
    while v != SOURCE:
      u = parent[v]
      path_flow = min(path_flow, capacity[u][v])
      v = u

    v = SINK
    while v != SOURCE:
      u = parent[v] # Changed U to u to match usage above
      capacity[u][v] -= path_flow
      capacity[v][u] += path_flow
      v = u
    parent = [-1]*N

  visited = [False] * N
  queue = [SOURCE]
  visited[SOURCE] = True
  while queue:
    u = queue.pop(0)
    for v in range(N):
      if not visited[v] and capacity[u][v] > 1e-9:
        visited[v] = True
        queue.append(v)
  labels = []
  for r in range(rows):
    row_labels = []
    for c in range(cols):
      node = r * cols + c
      row_labels.append(1 if visited[node] else 0)
    labels.append(row_labels)

  return labels



# ---------- Ejemplo ----------
def print_segmentation(image, labels):
    symbols = {0: '░', 1: '█'}
    print("Imagen original (intensidades):")
    for row in image:
        print("  " + " ".join(f"{v:3}" for v in row))
    print("\nSegmentación (█=frente, ░=fondo):")
    for row in labels:
        print("  " + " ".join(symbols[l] for l in row))


# Imagen 5x5 con objeto brillante en el centro
image = [
    [ 20,  25,  30,  25,  20],
    [ 25, 200, 210, 200,  25],
    [ 30, 210, 220, 210,  30],
    [ 25, 200, 210, 200,  25],
    [ 20,  25,  30,  25,  20],
]

# Foreground: píxeles brillantes tienen alta probabilidad de ser objeto
source_w = [[v / 255.0 for v in row] for row in image]
# Background: píxeles oscuros tienen alta probabilidad de ser fondo
sink_w   = [[(255 - v) / 255.0 for v in row] for row in image]

labels = image_segmentation_graph_cut(image, source_w, sink_w, smoothness=2.0)
print_segmentation(image, labels)

Imagen original (intensidades):
   20  25  30  25  20
   25 200 210 200  25
   30 210 220 210  30
   25 200 210 200  25
   20  25  30  25  20

Segmentación (█=frente, ░=fondo):
  ░ ░ ░ ░ ░
  ░ █ █ █ ░
  ░ █ █ █ ░
  ░ █ █ █ ░
  ░ ░ ░ ░ ░
